# Neutron Capture Probability Calculator

This notebook calculates the **probability of neutron capture** as a function of neutron energy for different materials, taking into account material properties and geometry.

## Features

⚛️ **Comprehensive Nuclear Database**: Includes thermal neutron cross-sections, resonance parameters, and fast neutron data for key isotopes

🎛️ **Configurable Parameters**: Easy selection of materials, density, thickness, and temperature

📊 **Multiple Plot Types**: Cross-sections, capture probabilities, thickness dependence, temperature effects

🌡️ **Temperature Effects**: Accurate 1/v law implementation with thermal motion corrections

🎯 **Interactive Analysis**: Functions for specific calculations and material comparisons

## Key Physics

The neutron capture probability is governed by:

$$P = 1 - e^{-\Sigma t}$$

Where:
- $\Sigma = n \sigma(E)$ = macroscopic cross-section (cm⁻¹)
- $n = \frac{\rho N_A \text{abundance}}{A}$ = number density (atoms/cm³)
- $\sigma(E)$ = energy-dependent microscopic cross-section (barns)
- $t$ = material thickness (cm)

### Cross-Section Models:
- **Thermal Region** (E < 0.5 eV): $\sigma(E) = \sigma_{th} \sqrt{\frac{E_{th}}{E}}$ (1/v law)
- **Resonance Region** (0.5 eV - 10 keV): Breit-Wigner resonance peaks
- **Fast Region** (E > 10 keV): Slowly varying cross-sections

---

**📝 Instructions**: Modify the configuration parameters, then run all cells to generate comprehensive neutron capture analysis!

In [9]:
"""
Neutron Capture Probability Calculator
=====================================

This notebook calculates the probability of neutron capture as a function of neutron energy
for different materials, taking into account material properties and geometry.

Key Physics:
- Capture probability: P = 1 - exp(-Σ * t) where Σ = n * σ(E)
- Number density: n = ρ * N_A * abundance / A  
- Cross-section σ(E) varies with energy (1/v law, resonances, etc.)
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

# Import physical constants - with fallback
try:
    from scipy.constants import N_A, pi
    print(f"✓ Avogadro's number from scipy: {N_A:.3e}")
except ImportError:
    # Fallback definitions
    N_A = 6.02214076e23  # mol⁻¹
    pi = 3.141592653589793
    print(f"✓ Using fallback constants: N_A = {N_A:.3e}")

# Try to import mendeleev for automatic isotope lookup
try:
    from mendeleev import element
    HAS_MENDELEEV = True
    print("✓ Mendeleev library available for automatic isotope lookup")
except ImportError:
    HAS_MENDELEEV = False
    print("⚠ Mendeleev library not available. Using built-in nuclear database")

print("✓ Neutron Capture Calculator libraries loaded successfully!")

ModuleNotFoundError: No module named 'scipy'

In [ ]:
# ===================================================================
# CONFIGURATION SECTION - MODIFY THESE PARAMETERS
# ===================================================================

# Select materials/isotopes to analyze
SELECTED_MATERIALS = [
    'B-10',      # Boron-10 (high thermal capture cross-section)
    'Cd-113',    # Cadmium-113 (strong neutron absorber)  
    'Gd-155',    # Gadolinium-155 (excellent thermal absorber)
    'Gd-157',    # Gadolinium-157 (highest known thermal cross-section)
    'Li-6',      # Lithium-6 (neutron converter)
    'He-3',      # Helium-3 (neutron detector gas)
    'U-235',     # Uranium-235 (fissile material)
    'H-1',       # Hydrogen (moderator)
]

# Material properties to vary
MATERIAL_DENSITIES = {
    'B-10': 2.34,       # g/cm³ (boron carbide)
    'Cd-113': 8.65,     # g/cm³ (metallic cadmium)
    'Gd-155': 7.90,     # g/cm³ (metallic gadolinium)  
    'Gd-157': 7.90,     # g/cm³ (metallic gadolinium)
    'Li-6': 0.534,      # g/cm³ (metallic lithium)
    'He-3': 0.000178,   # g/cm³ (gas at STP)
    'U-235': 19.05,     # g/cm³ (metallic uranium)
    'H-1': 1.0,         # g/cm³ (water)
}

# Geometric parameters
THICKNESSES = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]  # cm
DEFAULT_THICKNESS = 1.0  # cm

# Energy range for neutrons
ENERGY_MIN = 1e-9    # eV (ultra-cold neutrons)
ENERGY_MAX = 1e7     # eV (fast neutrons)
NUM_POINTS = 1000

# Temperature for thermal neutron calculations
TEMPERATURE = 300    # Kelvin (room temperature)

# Reference energy for normalization (thermal energy at room temp)
THERMAL_ENERGY = 0.0253  # eV at 20°C

print(f"Selected materials: {SELECTED_MATERIALS}")  
print(f"Energy range: {ENERGY_MIN:.1e} - {ENERGY_MAX:.1e} eV")
print(f"Default thickness: {DEFAULT_THICKNESS} cm")
print(f"Temperature: {TEMPERATURE} K (thermal energy = {THERMAL_ENERGY} eV)")

Selected materials: ['B-10', 'Cd-113', 'Gd-155', 'Gd-157', 'Li-6', 'He-3', 'U-235', 'H-1']
Energy range: 1.0e-09 - 1.0e+07 eV
Default thickness: 1.0 cm
Temperature: 300 K (thermal energy = 0.0253 eV)


In [ ]:
# ===================================================================
# NUCLEAR DATA DATABASE
# ===================================================================

class NuclearDatabase:
    """Database of neutron capture cross-sections and nuclear properties."""
    
    def __init__(self):
        # Nuclear data: {isotope: (mass, thermal_xs, abundance, resonances, fast_xs)}
        # mass: atomic mass (amu)
        # thermal_xs: thermal neutron capture cross-section (barns)
        # abundance: isotopic abundance (fraction)
        # resonances: [(E_res, Γ_n, Γ_γ, J), ...] - resonance parameters
        # fast_xs: fast neutron cross-section (barns)
        
        self.nuclear_data = {
            # Isotope: (mass, σ_thermal, abundance, resonances, σ_fast)
            'H-1': (1.008, 0.332, 0.99985, [], 0.0003),
            'He-3': (3.016, 5333, 0.000137, [], 0.0001),
            'Li-6': (6.015, 940, 0.0759, [], 0.0009),
            'B-10': (10.013, 3835, 0.199, [(0.0025, 0.0001, 0.0024, 2)], 0.0005),
            'Cd-113': (112.904, 20600, 0.122, [(0.178, 0.00011, 0.116, 1)], 0.001),
            'Gd-155': (154.923, 60900, 0.148, [(0.0268, 0.00004, 0.108, 2)], 0.002),
            'Gd-157': (156.924, 254000, 0.157, [(0.0314, 0.00006, 0.106, 2)], 0.002),
            'U-235': (235.044, 680.9, 0.0072, [(0.29, 0.0015, 0.037, 3)], 0.0045),
            
            # Additional common isotopes
            'C-12': (12.000, 0.00353, 0.9893, [], 0.0001),
            'Al-27': (26.982, 0.231, 1.0, [], 0.0009),
            'Fe-56': (55.845, 2.59, 0.9175, [], 0.0011),
            'Ag-107': (106.905, 37.6, 0.5184, [], 0.0063),
            'In-115': (114.904, 202, 0.957, [], 0.0012),
        }
        
        # Physical constants
        self.barn = 1e-24  # cm²
        self.avogadro = N_A
        
    def get_isotope_data(self, isotope):
        """Get nuclear data for a specific isotope."""
        if isotope not in self.nuclear_data:
            available = list(self.nuclear_data.keys())
            raise ValueError(f"Isotope {isotope} not found. Available: {available}")
        
        return self.nuclear_data[isotope]
    
    def calculate_number_density(self, isotope, density_g_cm3, abundance_override=None):
        """Calculate number density of nuclei in atoms/cm³."""
        mass, thermal_xs, abundance, resonances, fast_xs = self.get_isotope_data(isotope)
        
        if abundance_override is not None:
            abundance = abundance_override
            
        # Number density = ρ * N_A * abundance / A
        number_density = density_g_cm3 * self.avogadro * abundance / mass
        return number_density
    
    def thermal_cross_section(self, isotope, energy_ev, temperature_k=300):
        """Calculate thermal neutron cross-section using 1/v law."""
        mass, thermal_xs, abundance, resonances, fast_xs = self.get_isotope_data(isotope)
        
        # Thermal energy at given temperature
        k_b = 8.617333e-5  # eV/K
        thermal_energy_T = k_b * temperature_k
        
        # 1/v law: σ(E) = σ_th * sqrt(E_th/E) * sqrt(T_ref/T)
        thermal_factor = np.sqrt(THERMAL_ENERGY / energy_ev)
        temperature_factor = np.sqrt(300 / temperature_k)
        
        return thermal_xs * thermal_factor * temperature_factor
    
    def resonance_cross_section(self, isotope, energy_ev):
        """Calculate resonance contribution to cross-section."""
        mass, thermal_xs, abundance, resonances, fast_xs = self.get_isotope_data(isotope)
        
        if not resonances:
            return np.zeros_like(energy_ev)
        
        total_resonance = np.zeros_like(energy_ev)
        
        for E_res, Gamma_n, Gamma_gamma, J in resonances:
            # Breit-Wigner formula (simplified)
            Gamma_total = Gamma_n + Gamma_gamma
            
            # Statistical factor
            g = (2*J + 1) / 4  # Assuming spin 1/2 neutrons, spin 0 target for simplicity
            
            # Breit-Wigner peak
            sigma_res = g * (Gamma_n * Gamma_gamma) / ((energy_ev - E_res)**2 + (Gamma_total/2)**2)
            sigma_res *= 4 * np.pi / (2 * 0.0253)  # Normalization factor
            
            total_resonance += sigma_res
            
        return total_resonance
    
    def calculate_capture_cross_section(self, isotope, energy_ev, temperature_k=300):
        """Calculate total capture cross-section vs energy."""
        mass, thermal_xs, abundance, resonances, fast_xs = self.get_isotope_data(isotope)
        
        # Convert energy to array if needed
        energy_ev = np.array(energy_ev, dtype=float)
        
        # Initialize cross-section array
        sigma_total = np.zeros_like(energy_ev)
        
        # Thermal region (E < 0.5 eV)
        thermal_mask = energy_ev < 0.5
        if np.any(thermal_mask):
            sigma_total[thermal_mask] = self.thermal_cross_section(
                isotope, energy_ev[thermal_mask], temperature_k)
        
        # Resonance region (0.5 eV < E < 10 keV)
        resonance_mask = (energy_ev >= 0.5) & (energy_ev < 1e4)
        if np.any(resonance_mask):
            # Smooth transition from thermal + resonances
            thermal_part = self.thermal_cross_section(
                isotope, energy_ev[resonance_mask], temperature_k) * 0.1  # Reduced thermal
            resonance_part = self.resonance_cross_section(
                isotope, energy_ev[resonance_mask])
            sigma_total[resonance_mask] = thermal_part + resonance_part
        
        # Fast neutron region (E > 10 keV)
        fast_mask = energy_ev >= 1e4
        if np.any(fast_mask):
            # Transition to constant fast cross-section
            transition_energy = 1e4
            transition_xs = self.thermal_cross_section(isotope, transition_energy, temperature_k) * 0.1
            
            # Linear interpolation to fast cross-section
            log_energy_fast = np.log10(energy_ev[fast_mask])
            log_transition = np.log10(transition_energy)
            log_fast_energy = np.log10(1e6)  # 1 MeV
            
            interp_factor = np.clip((log_energy_fast - log_transition) / (log_fast_energy - log_transition), 0, 1)
            sigma_total[fast_mask] = transition_xs * (1 - interp_factor) + fast_xs * interp_factor
        
        return sigma_total
    
    def list_available_isotopes(self):
        """List all available isotopes with their thermal cross-sections."""
        print("Available isotopes and thermal capture cross-sections (barns):")
        print("-" * 60)
        for isotope, (mass, thermal_xs, abundance, resonances, fast_xs) in self.nuclear_data.items():
            print(f"{isotope:8s} | σ_th = {thermal_xs:8.1f} b | abundance = {abundance:6.4f}")

# Initialize nuclear database
nuclear_db = NuclearDatabase()
nuclear_db.list_available_isotopes()

NameError: name 'N_A' is not defined

In [ ]:
# ===================================================================
# NEUTRON CAPTURE CALCULATION FUNCTIONS
# ===================================================================

def calculate_capture_probability(energies_ev, isotope, density_g_cm3=None, 
                                thickness_cm=DEFAULT_THICKNESS, temperature_k=TEMPERATURE,
                                abundance_override=None):
    """
    Calculate neutron capture probability as function of energy.
    
    Parameters:
    -----------
    energies_ev : array-like
        Neutron energies in eV
    isotope : str
        Isotope name (e.g., 'B-10', 'Gd-157')
    density_g_cm3 : float
        Material density in g/cm³ (uses default if None)
    thickness_cm : float
        Material thickness in cm
    temperature_k : float
        Temperature in Kelvin
    abundance_override : float
        Override natural abundance (0-1)
        
    Returns:
    --------
    probability : array
        Capture probability (0-1)
    cross_section : array
        Microscopic cross-section in barns
    macro_cross_section : array  
        Macroscopic cross-section in cm⁻¹
    """
    
    # Use default density if not specified
    if density_g_cm3 is None:
        if isotope in MATERIAL_DENSITIES:
            density_g_cm3 = MATERIAL_DENSITIES[isotope]
        else:
            density_g_cm3 = 1.0  # Default fallback
            print(f"Warning: Using default density 1.0 g/cm³ for {isotope}")
    
    # Convert energies to array
    energies_ev = np.array(energies_ev, dtype=float)
    
    # Calculate microscopic cross-section
    cross_section = nuclear_db.calculate_capture_cross_section(
        isotope, energies_ev, temperature_k)
    
    # Calculate number density
    number_density = nuclear_db.calculate_number_density(
        isotope, density_g_cm3, abundance_override)
    
    # Macroscopic cross-section (cm⁻¹)
    macro_cross_section = number_density * cross_section * nuclear_db.barn
    
    # Capture probability
    probability = 1.0 - np.exp(-macro_cross_section * thickness_cm)
    
    return probability, cross_section, macro_cross_section

def calculate_transmission(energies_ev, isotope, density_g_cm3=None,
                         thickness_cm=DEFAULT_THICKNESS, temperature_k=TEMPERATURE):
    """Calculate neutron transmission probability (1 - capture probability)."""
    prob, xs, macro_xs = calculate_capture_probability(
        energies_ev, isotope, density_g_cm3, thickness_cm, temperature_k)
    return 1.0 - prob, xs, macro_xs

def calculate_specific_case(isotope, energy_ev, density_g_cm3=None, 
                          thickness_cm=DEFAULT_THICKNESS, temperature_k=TEMPERATURE):
    """Calculate and display capture probability for a specific case."""
    
    # Use default density if not specified
    if density_g_cm3 is None and isotope in MATERIAL_DENSITIES:
        density_g_cm3 = MATERIAL_DENSITIES[isotope]
    
    try:
        prob, xs, macro_xs = calculate_capture_probability(
            [energy_ev], isotope, density_g_cm3, thickness_cm, temperature_k)
        
        print(f"\n{'='*60}")
        print(f"NEUTRON CAPTURE CALCULATION")
        print(f"{'='*60}")
        print(f"Isotope: {isotope}")
        print(f"Neutron Energy: {energy_ev:.6f} eV")
        print(f"Material Density: {density_g_cm3:.3f} g/cm³")
        print(f"Thickness: {thickness_cm:.2f} cm")
        print(f"Temperature: {temperature_k:.1f} K")
        print(f"")
        print(f"Results:")
        print(f"  Microscopic σ: {xs[0]:.3f} barns")
        print(f"  Macroscopic Σ: {macro_xs[0]:.6f} cm⁻¹")  
        print(f"  Capture Probability: {prob[0]:.6f} ({prob[0]*100:.4f}%)")
        print(f"  Transmission: {(1-prob[0]):.6f} ({(1-prob[0])*100:.4f}%)")
        
        # Material info
        mass, thermal_xs, abundance, resonances, fast_xs = nuclear_db.get_isotope_data(isotope)
        number_density = nuclear_db.calculate_number_density(isotope, density_g_cm3)
        
        print(f"")
        print(f"Material Properties:")
        print(f"  Atomic Mass: {mass:.3f} amu")
        print(f"  Natural Abundance: {abundance:.4f}")
        print(f"  Thermal σ (0.0253 eV): {thermal_xs:.1f} barns")
        print(f"  Number Density: {number_density:.3e} atoms/cm³")
        print(f"{'='*60}")
        
        return prob[0], xs[0], macro_xs[0]
        
    except Exception as e:
        print(f"Error in calculation: {e}")
        return None, None, None

def compare_materials_at_energy(energy_ev, materials_list, thickness_cm=DEFAULT_THICKNESS):
    """Compare capture probabilities for different materials at fixed energy."""
    
    print(f"\nMaterial Comparison at {energy_ev} eV (thickness = {thickness_cm} cm)")
    print("-" * 80)
    print(f"{'Isotope':<12} {'Density':<10} {'σ (barns)':<12} {'Σ (cm⁻¹)':<12} {'P_capture':<12}")
    print("-" * 80)
    
    results = []
    
    for isotope in materials_list:
        try:
            density = MATERIAL_DENSITIES.get(isotope, 1.0)
            prob, xs, macro_xs = calculate_capture_probability(
                [energy_ev], isotope, density, thickness_cm)
            
            print(f"{isotope:<12} {density:<10.3f} {xs[0]:<12.3f} {macro_xs[0]:<12.6f} {prob[0]:<12.6f}")
            results.append((isotope, prob[0], xs[0], macro_xs[0]))
            
        except Exception as e:
            print(f"{isotope:<12} Error: {str(e)[:50]}")
    
    print("-" * 80)
    return results

print("✓ Neutron capture calculation functions loaded")

In [ ]:
# ===================================================================
# PLOTTING FUNCTIONS
# ===================================================================

def plot_cross_sections(materials_list, energies, temperature_k=TEMPERATURE):
    """Plot neutron capture cross-sections vs energy for different isotopes."""
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(materials_list)))
    
    for isotope, color in zip(materials_list, colors):
        try:
            cross_section = nuclear_db.calculate_capture_cross_section(
                isotope, energies, temperature_k)
            
            # Plot only positive cross-sections
            valid_mask = cross_section > 0
            if np.any(valid_mask):
                plt.loglog(energies[valid_mask], cross_section[valid_mask],
                          label=isotope, color=color, linewidth=2.5)
        except Exception as e:
            print(f"Warning: Could not plot {isotope}: {e}")
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)
    plt.ylabel('Capture Cross-Section (barns)', fontsize=14)
    plt.title('Neutron Capture Cross-Sections vs Energy', fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_capture_probabilities(materials_list, energies, thickness_cm=DEFAULT_THICKNESS,
                             temperature_k=TEMPERATURE):
    """Plot capture probabilities vs energy for different materials."""
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(materials_list)))
    
    for isotope, color in zip(materials_list, colors):
        try:
            density = MATERIAL_DENSITIES.get(isotope, 1.0)
            prob, xs, macro_xs = calculate_capture_probability(
                energies, isotope, density, thickness_cm, temperature_k)
            
            # Plot all probabilities
            plt.loglog(energies, prob, label=f'{isotope} ({thickness_cm} cm)',
                      color=color, linewidth=2.5)
            
        except Exception as e:
            print(f"Warning: Could not plot {isotope}: {e}")
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)
    plt.ylabel('Capture Probability', fontsize=14)
    plt.title(f'Neutron Capture Probability vs Energy (thickness = {thickness_cm} cm)', fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.ylim(1e-6, 1)
    plt.tight_layout()
    plt.show()

def plot_thickness_dependence(isotope, energies_to_plot, thickness_range=None,
                            temperature_k=TEMPERATURE):
    """Plot capture probability vs thickness for different neutron energies."""
    
    if thickness_range is None:
        thickness_range = np.logspace(-2, 2, 100)  # 0.01 to 100 cm
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(energies_to_plot)))
    
    density = MATERIAL_DENSITIES.get(isotope, 1.0)
    
    for energy_ev, color in zip(energies_to_plot, colors):
        try:
            prob_vs_thickness = []
            for thickness in thickness_range:
                prob, xs, macro_xs = calculate_capture_probability(
                    [energy_ev], isotope, density, thickness, temperature_k)
                prob_vs_thickness.append(prob[0])
            
            plt.semilogx(thickness_range, prob_vs_thickness,
                        label=f'{energy_ev} eV', color=color, linewidth=2.5)
            
        except Exception as e:
            print(f"Warning: Could not plot energy {energy_ev}: {e}")
    
    plt.xlabel('Thickness (cm)', fontsize=14)
    plt.ylabel('Capture Probability', fontsize=14)
    plt.title(f'Capture Probability vs Thickness - {isotope}', fontsize=16)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

def plot_temperature_dependence(isotope, energies, temperatures=[77, 300, 600, 1000],
                              thickness_cm=DEFAULT_THICKNESS):
    """Plot how temperature affects capture probability."""
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.plasma(np.linspace(0, 1, len(temperatures)))
    density = MATERIAL_DENSITIES.get(isotope, 1.0)
    
    for temp_k, color in zip(temperatures, colors):
        try:
            prob, xs, macro_xs = calculate_capture_probability(
                energies, isotope, density, thickness_cm, temp_k)
            
            plt.loglog(energies, prob, label=f'{temp_k} K',
                      color=color, linewidth=2.5)
            
        except Exception as e:
            print(f"Warning: Could not plot temperature {temp_k}: {e}")
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)
    plt.ylabel('Capture Probability', fontsize=14)
    plt.title(f'Temperature Dependence - {isotope} ({thickness_cm} cm thick)', fontsize=16)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_neutron_spectra_comparison(isotope, spectra_dict, thickness_cm=DEFAULT_THICKNESS):
    """Plot capture probability for different neutron spectra.
    
    spectra_dict: {'name': (energies, flux_weights), ...}
    """
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(spectra_dict)))
    density = MATERIAL_DENSITIES.get(isotope, 1.0)
    
    for (spectrum_name, (energies, weights)), color in zip(spectra_dict.items(), colors):
        try:
            prob, xs, macro_xs = calculate_capture_probability(
                energies, isotope, density, thickness_cm)
            
            # Weight the probabilities by flux
            weighted_prob = prob * weights / np.sum(weights)
            
            plt.loglog(energies, weighted_prob, label=spectrum_name,
                      color=color, linewidth=2.5)
            
        except Exception as e:
            print(f"Warning: Could not plot spectrum {spectrum_name}: {e}")
    
    plt.xlabel('Neutron Energy (eV)', fontsize=14)  
    plt.ylabel('Weighted Capture Probability', fontsize=14)
    plt.title(f'Capture Probability for Different Neutron Spectra - {isotope}', fontsize=16)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def interactive_comparison():
    """Interactive function to compare materials at user-specified energy."""
    print("\nInteractive Material Comparison")
    print("Available materials:", SELECTED_MATERIALS)
    
    try:
        energy_str = input("Enter neutron energy (eV): ")
        energy_ev = float(energy_str)
        
        thickness_str = input(f"Enter thickness (cm, default {DEFAULT_THICKNESS}): ")
        thickness_cm = float(thickness_str) if thickness_str else DEFAULT_THICKNESS
        
        results = compare_materials_at_energy(energy_ev, SELECTED_MATERIALS, thickness_cm)
        
        # Plot the results
        materials = [r[0] for r in results]
        probabilities = [r[1] for r in results]
        
        plt.figure(figsize=(10, 6))
        bars = plt.bar(materials, probabilities, color=plt.cm.viridis(np.linspace(0, 1, len(materials))))
        plt.ylabel('Capture Probability', fontsize=12)
        plt.title(f'Material Comparison at {energy_ev} eV ({thickness_cm} cm thick)', fontsize=14)
        plt.xticks(rotation=45)
        
        # Add value labels on bars
        for bar, prob in zip(bars, probabilities):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{prob:.4f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error: {e}")

print("✓ Plotting functions loaded")

In [ ]:
# ===================================================================
# GENERATE ENERGY ARRAY AND SETUP
# ===================================================================

# Generate neutron energy array (logarithmic spacing)
energies = np.logspace(np.log10(ENERGY_MIN), np.log10(ENERGY_MAX), NUM_POINTS)

print(f"Generated {NUM_POINTS} energy points from {ENERGY_MIN:.1e} to {ENERGY_MAX:.1e} eV")
print(f"Temperature: {TEMPERATURE} K")
print(f"Default thickness: {DEFAULT_THICKNESS} cm")

# Selected materials info
print(f"\nSelected materials for analysis:")
print("-" * 50) 
for i, isotope in enumerate(SELECTED_MATERIALS, 1):
    try:
        density = MATERIAL_DENSITIES.get(isotope, 'N/A')
        mass, thermal_xs, abundance, resonances, fast_xs = nuclear_db.get_isotope_data(isotope)
        print(f"{i:2d}. {isotope:8s} | σ_th = {thermal_xs:8.1f} b | ρ = {density:6.3f} g/cm³")
    except Exception as e:
        print(f"{i:2d}. {isotope:8s} - Error: {e}")

print(f"\n✓ Ready to calculate neutron capture for {len(SELECTED_MATERIALS)} isotopes!")

In [ ]:
# ===================================================================
# PLOT 1: NEUTRON CAPTURE CROSS-SECTIONS
# ===================================================================

# Plot cross-sections vs energy for all selected materials
print("Plotting neutron capture cross-sections...")
plot_cross_sections(SELECTED_MATERIALS, energies, TEMPERATURE)

In [ ]:
# ===================================================================
# PLOT 2: CAPTURE PROBABILITIES VS ENERGY
# ===================================================================

# Plot capture probabilities for different materials
print("Plotting neutron capture probabilities...")
plot_capture_probabilities(SELECTED_MATERIALS, energies, DEFAULT_THICKNESS, TEMPERATURE)

In [ ]:
# ===================================================================
# PLOT 3: THICKNESS DEPENDENCE EXAMPLE
# ===================================================================

# Example: Boron-10 thickness dependence at different energies
example_energies = [0.01, 0.0253, 0.1, 1.0, 100.0]  # eV 

print("Plotting thickness dependence for B-10...")
plot_thickness_dependence('B-10', example_energies)

In [ ]:
# ===================================================================
# PLOT 4: TEMPERATURE DEPENDENCE EXAMPLE
# ===================================================================

# Example: Gadolinium-157 temperature dependence
print("Plotting temperature dependence for Gd-157...")
plot_temperature_dependence('Gd-157', energies, [77, 300, 600, 1000], DEFAULT_THICKNESS)

In [ ]:
# ===================================================================
# EXAMPLE CALCULATIONS
# ===================================================================

# Specific calculation examples
print("EXAMPLE SPECIFIC CALCULATIONS")
print("=" * 70)

# Example cases: (isotope, energy_eV, thickness_cm)
example_cases = [
    ("Gd-157", 0.0253, 1.0),    # Thermal neutrons in Gd
    ("B-10", 0.0253, 0.1),      # Thermal neutrons in thin B layer  
    ("Cd-113", 0.178, 2.0),     # Resonance energy in Cd
    ("He-3", 0.0253, 10.0),     # Thermal neutrons in He-3 detector
    ("U-235", 0.29, 5.0),       # Near resonance in U-235
]

for isotope, energy, thickness in example_cases:
    calculate_specific_case(isotope, energy, thickness_cm=thickness)

In [ ]:
# ===================================================================
# MATERIAL COMPARISON AT SPECIFIC ENERGIES
# ===================================================================

# Compare all materials at thermal energy
print("\nMaterial comparison at thermal energy (0.0253 eV):")
thermal_results = compare_materials_at_energy(0.0253, SELECTED_MATERIALS, 1.0)

print("\nMaterial comparison at fast neutron energy (1 MeV):")
fast_results = compare_materials_at_energy(1e6, SELECTED_MATERIALS, 1.0)

In [ ]:
# ===================================================================
# USAGE INSTRUCTIONS AND CUSTOMIZATION
# ===================================================================

print("\n" + "="*80)
print("NEUTRON CAPTURE CALCULATOR - READY FOR USE!")
print("="*80)

print("\n🎯 HOW TO USE THIS NOTEBOOK:")
print("1. Modify SELECTED_MATERIALS list to choose isotopes")
print("2. Adjust MATERIAL_DENSITIES, THICKNESSES, and energy range") 
print("3. Run all cells to generate comprehensive plots")
print("4. Use specific calculation functions for detailed analysis")

print("\n📚 AVAILABLE FUNCTIONS:")
print("• calculate_capture_probability(energies, isotope, density, thickness, temp)")
print("• calculate_specific_case(isotope, energy_ev, density, thickness, temp)")
print("• compare_materials_at_energy(energy_ev, materials_list, thickness)")
print("• plot_cross_sections(materials_list, energies, temperature)")
print("• plot_capture_probabilities(materials_list, energies, thickness, temp)")
print("• plot_thickness_dependence(isotope, energies_list, thickness_range)")
print("• plot_temperature_dependence(isotope, energies, temperatures, thickness)")
print("• interactive_comparison()")

print("\n🧪 CONFIGURABLE PARAMETERS:")
print("• Material/Isotope: Choose from nuclear database or add new ones")
print("• Density: Material density in g/cm³")  
print("• Thickness: Absorber thickness in cm")
print("• Temperature: Affects 1/v law for thermal neutrons (Kelvin)")
print("• Energy Range: From ultra-cold to fast neutrons (eV)")
print("• Abundance: Override natural isotopic abundances")

print("\n⚛️ PHYSICS INCLUDED:")
print("• 1/v law for thermal neutrons with temperature dependence")
print("• Breit-Wigner resonances for major peaks")
print("• Smooth transition to fast neutron cross-sections")  
print("• Full Beer-Lambert attenuation: P = 1 - exp(-Σt)")
print("• Accurate number densities: n = ρ·N_A·abundance/A")

print("\n🚀 TO ADD NEW ISOTOPES:")
print("• Add entry to nuclear_data dictionary in NuclearDatabase class")
print("• Format: 'Isotope': (mass, σ_thermal, abundance, resonances, σ_fast)")
print("• Resonances: [(E_res, Γ_n, Γ_γ, J), ...] for Breit-Wigner peaks")

print("\n💡 EXAMPLE USAGE:")
print("# Calculate B-10 capture at thermal energy")
print("prob, xs, macro_xs = calculate_capture_probability([0.0253], 'B-10', 2.34, 0.1)")
print("print(f'Capture probability: {prob[0]:.4f}')")

print("\n# Interactive comparison")
print("# interactive_comparison()  # Uncomment to run")

print("\n" + "="*80)
print("🔬 Perfect for neutron detector design, shielding calculations,")
print("   reactor physics, and nuclear engineering applications!")
print("="*80)